In [ ]:
import psycopg2
import numpy as np
import time

from extractor.pdf_text_extractor import pdf_file_data_extract
from chunkers.recursive_splitter import recursive_splitter
from embeddings.gemini_embedder import gemini_embedder
# from embeddings.hugging_face_embedder import hugging_face_embedder

In [34]:
docx_file_path = "./documents/AI_Learning_Plan.docx"
pdf_file_path = "./documents/OS_Full_Notes.pdf"

file_name = pdf_file_path.split('/')[-1]

extracted_text = pdf_file_data_extract(pdf_file_path)

<class 'str'>


In [35]:
chunks = recursive_splitter(extracted_text)

In [ ]:
batch_size = 100
rpm_limit = 100  # Gemini free tier = 100 requests per minute

all_embeddings = []
request_count = 0

try:

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        print(f"Embedding batch, chunks {i} to {i + len(batch)}...")

        result = gemini_embedder(batch)
        all_embeddings.extend(result)

        request_count += 1

        # Once we hit the RPM limit, pause for a minute
        if request_count >= rpm_limit:
            print("Rate limit reached. Waiting 65s...")
            time.sleep(65)
            request_count = 0  # reset counter

    print(f"Done! Total embeddings: {len(all_embeddings)}")

except Exception as e:

    print(e)

Embedding batch, chunks 0 to 100...
Embedding batch, chunks 100 to 137...


AttributeError: 'Response' object has no attribute 'body'

In [ ]:
{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 57.901106513s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-embedding-1.0'}, 'quotaValue': '100'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '57s'}]}}

In [28]:
len(chunks)

737

In [32]:
import time

batch_size = 1
rpm_limit = 100  # Gemini free tier = 100 requests per minute

all_embeddings = []
request_count = 0

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"Embedding batch, chunks {i} to {i + len(batch)}...")

    # result = embeddings_model.embed_documents(batch)
    # all_embeddings.extend(result)

    request_count += 1

    # Once we hit the RPM limit, pause for a minute
    if request_count >= rpm_limit:
        print("Rate limit reached. Waiting 65s...")
        time.sleep(65)
        request_count = 0  # reset counter

print(f"Done! Total embeddings: {len(all_embeddings)}")

Embedding batch, chunks 0 to 1...
Embedding batch, chunks 1 to 2...
Embedding batch, chunks 2 to 3...
Embedding batch, chunks 3 to 4...
Embedding batch, chunks 4 to 5...
Embedding batch, chunks 5 to 6...
Embedding batch, chunks 6 to 7...
Embedding batch, chunks 7 to 8...
Embedding batch, chunks 8 to 9...
Embedding batch, chunks 9 to 10...
Embedding batch, chunks 10 to 11...
Embedding batch, chunks 11 to 12...
Embedding batch, chunks 12 to 13...
Embedding batch, chunks 13 to 14...
Embedding batch, chunks 14 to 15...
Embedding batch, chunks 15 to 16...
Embedding batch, chunks 16 to 17...
Embedding batch, chunks 17 to 18...
Embedding batch, chunks 18 to 19...
Embedding batch, chunks 19 to 20...
Embedding batch, chunks 20 to 21...
Embedding batch, chunks 21 to 22...
Embedding batch, chunks 22 to 23...
Embedding batch, chunks 23 to 24...
Embedding batch, chunks 24 to 25...
Embedding batch, chunks 25 to 26...
Embedding batch, chunks 26 to 27...
Embedding batch, chunks 27 to 28...
Embedding b

KeyboardInterrupt: 

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",  # lightweight, ~130MB
    model_kwargs={"device": "cpu"}
)

def hugging_face_embedder_local(chunks):  # same interface, drop-in replacement
    return embeddings.embed_documents(chunks)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1148.92it/s]


In [ ]:
embeddings.save("./newmodel")

AttributeError: 'HuggingFaceEmbeddings' object has no attribute 'save'

In [ ]:
vectors = hugging_face_embedder_local(chunks)

KeyboardInterrupt: 

In [ ]:
len(vectors[0])

1024